# TabICL vs CatBoost: Foundation Model vs Gradient Boosting

## Que es TabICL?

**TabICL** (Tabular In-Context Learning) es un **foundation model** para datos tabulares
presentado en **ICML 2025** por el equipo SODA de INRIA (Francia).
([paper](https://arxiv.org/abs/2502.05564),
[v2](https://arxiv.org/abs/2602.11139))

A diferencia de los modelos clasicos (CatBoost, XGBoost, etc.) que entrenan arboles
sobre nuestros datos, TabICL funciona mediante **in-context learning**:

1. Toma las filas de entrenamiento etiquetadas como "contexto"
2. Predice sobre nuevas filas en un **unico forward pass**, sin entrenar ni actualizar gradientes
3. Usa una arquitectura transformer de dos etapas: atencion columna-fila + transformer ICL

**Resultados destacados:**
- **#1 en log-loss y Brier score** entre 21 clasificadores en el benchmark TabArena-v0.1
  (Manokhin & Gronhaug, "Classifier Calibration at Scale")
- Supera a CatBoost y TabPFN v2 en datasets con >10K muestras
- Hasta 10x mas rapido que TabPFN v2
- Pre-entrenado en datos sinteticos, sin necesidad de fine-tuning

## Objetivo de este notebook

Comparar TabICL con CatBoost (nuestro mejor modelo del notebook 07) usando el mismo
pipeline de preprocesamiento y filtrado, evaluando: AUC-ROC, F1, Recall y calibracion.

## PASO 1: Preprocesamiento y Filtrado (mismo pipeline que 05/06/07)

In [1]:
from src.preprocessing.base_preprocessing import BasePreprocess
from src.filtering.base_filtering import BaseFiltering
import numpy as np

# Preprocesamiento
base_pre = BasePreprocess("data/variables_withoutExperts.xlsx", "loan_status")
base_pre.fit("data/df_train_small.csv")

X_train, y_train = base_pre.transform("data/df_train_small.csv")
X_test, y_test = base_pre.transform("data/df_test_small.csv")

# Filtrado
base_filter = BaseFiltering(
    constant_tol=0.9,
    correlation_threshold=0.8,
    probe_n_probes=10,
    probe_scoring='roc_auc',
    probe_cv=3,
    probe_n_estimators=50,
    probe_max_depth=10
)
base_filter.fit(X_train, y_train)
base_filter.print_summary()

X_train_filtered = base_filter.transform(X_train)
X_test_filtered = base_filter.transform(X_test)

y_train_flat = y_train.values.ravel()
y_test_flat = y_test.values.ravel()

print(f"\nTrain: {X_train_filtered.shape}, Test: {X_test_filtered.shape}")
print(f"Defaults en train: {y_train_flat.sum()} ({y_train_flat.mean()*100:.1f}%)")
print(f"Defaults en test:  {y_test_flat.sum()} ({y_test_flat.mean()*100:.1f}%)")

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/src/preprocessing/base_preprocessing.py:62: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.train_X_data['earliest_cr_line'] = pd.to_datetime(self.train_X_data['earliest_cr_line'])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RESUMEN DEL PIPELINE DE FILTRADO
  Features iniciales:              2614
  Eliminadas cuasi-constantes:     -133
  Eliminadas por correlacion:       -1840
  Eliminadas por ProbeFeature:      -428
  Features seleccionadas finales:  213



Train: (80000, 213), Test: (20000, 213)
Defaults en train: 16236 (20.3%)
Defaults en test:  3997 (20.0%)


In [2]:
# Funcion de evaluacion comun
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt

def evaluate_model(model, X_test, y_test, model_name, results_dict):
    """Evalua un modelo y guarda metricas en results_dict."""
    class_pred = model.predict(X_test)
    prob_pred = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, class_pred)
    prec = precision_score(y_test, class_pred)
    rec = recall_score(y_test, class_pred)
    f1 = f1_score(y_test, class_pred)
    auc = roc_auc_score(y_test, prob_pred)

    results_dict[model_name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'F1-Score': f1, 'AUC-ROC': auc,
        'prob_pred': prob_pred, 'class_pred': class_pred
    }

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(classification_report(y_test, class_pred,
                                target_names=["Fully Paid", "Default"]))
    print(f"  AUC-ROC: {auc:.4f}")
    return results_dict

results = {}

## PASO 2: CatBoost (referencia del notebook 07)

Usamos los mismos parametros de mejores practicas del notebook anterior.

In [3]:
from catboost import CatBoostClassifier

catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    min_data_in_leaf=20,
    subsample=0.8,
    colsample_bylevel=0.8,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=200,
    early_stopping_rounds=50,
    eval_metric='AUC'
)

catboost_model.fit(
    X_train_filtered, y_train_flat,
    eval_set=(X_test_filtered, y_test_flat),
    verbose=200
)

results = evaluate_model(catboost_model, X_test_filtered, y_test_flat, "CatBoost", results)

0:	test: 0.6586873	best: 0.6586873 (0)	total: 87.6ms	remaining: 1m 27s


200:	test: 0.7048032	best: 0.7048207 (199)	total: 3.57s	remaining: 14.2s


400:	test: 0.7071036	best: 0.7075465 (360)	total: 6.74s	remaining: 10.1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7075465163
bestIteration = 360

Shrink model to first 361 iterations.



  CatBoost
              precision    recall  f1-score   support

  Fully Paid       0.88      0.67      0.76     16003
     Default       0.32      0.63      0.43      3997

    accuracy                           0.66     20000
   macro avg       0.60      0.65      0.59     20000
weighted avg       0.77      0.66      0.69     20000

  AUC-ROC: 0.7075


## PASO 3: TabICL

TabICL es compatible con la API de scikit-learn (`fit`/`predict`/`predict_proba`).
No requiere ajuste de hiperparametros: el modelo viene pre-entrenado y funciona
directamente con los datos.

**Parametros utilizados:**
- `n_estimators=8` (default): numero de miembros del ensemble. Cada uno aplica
  una permutacion diferente de features y clases para mayor robustez
- `softmax_temperature=0.9` (default): temperatura del softmax para suavizar
  las probabilidades predichas
- `device='cpu'`: en Mac sin GPU CUDA usamos CPU. En GPU H100 seria ~10x mas rapido
- `random_state=42`: reproducibilidad

> **Nota:** TabICL descarga automaticamente el checkpoint pre-entrenado (~100MB)
> desde Hugging Face Hub en el primer uso.

In [4]:
from tabicl import TabICLClassifier
import time

tabicl_model = TabICLClassifier(
    n_estimators=8,
    device='cpu',
    random_state=42,
    verbose=True
)

# TabICL: fit = almacenar contexto, predict = forward pass del transformer
print("Entrenando TabICL (almacenando contexto)...")
start = time.time()
tabicl_model.fit(X_train_filtered.values, y_train_flat.astype(int))
fit_time = time.time() - start
print(f"Fit completado en {fit_time:.1f}s")

print("\nPrediciendo...")
start = time.time()
tabicl_proba = tabicl_model.predict_proba(X_test_filtered.values)
tabicl_pred = tabicl_model.predict(X_test_filtered.values)
pred_time = time.time() - start
print(f"Prediccion completada en {pred_time:.1f}s")

Entrenando TabICL (almacenando contexto)...
INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.



tabicl-classifier-v2-20260212.ckpt:   0%|          | 0.00/110M [00:00<?, ?B/s]

Fit completado en 61.8s

Prediciendo...


In [ ]:
# Evaluar TabICL manualmente (predict ya hecho arriba)
acc = accuracy_score(y_test_flat, tabicl_pred)
prec = precision_score(y_test_flat, tabicl_pred)
rec = recall_score(y_test_flat, tabicl_pred)
f1 = f1_score(y_test_flat, tabicl_pred)
auc = roc_auc_score(y_test_flat, tabicl_proba[:, 1])

results["TabICL"] = {
    'Accuracy': acc, 'Precision': prec, 'Recall': rec,
    'F1-Score': f1, 'AUC-ROC': auc,
    'prob_pred': tabicl_proba[:, 1], 'class_pred': tabicl_pred
}

print(f"\n{'='*60}")
print(f"  TabICL")
print(f"{'='*60}")
print(classification_report(y_test_flat, tabicl_pred,
                            target_names=["Fully Paid", "Default"]))
print(f"  AUC-ROC: {auc:.4f}")

## PASO 4: Comparacion directa

In [ ]:
import pandas as pd

# Tabla comparativa
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
comparison = pd.DataFrame({
    name: {m: vals[m] for m in metrics_cols}
    for name, vals in results.items()
}).T

# Anadir baselines de notebooks anteriores
comparison.loc['Baseline FICO'] = [0.72, 0.26, 0.24, 0.25, 0.5921]
comparison.loc['Random Forest (05_)'] = [0.80, 0.5093, 0.0205, 0.0394, 0.6829]

comparison = comparison.sort_values('AUC-ROC', ascending=False)

print("=" * 80)
print("COMPARACION DE MODELOS (ordenados por AUC-ROC)")
print("=" * 80)
comparison.style.format("{:.4f}").highlight_max(axis=0, color='lightgreen')

In [ ]:
# Curvas ROC
plt.figure(figsize=(10, 7))

colors = {'CatBoost': '#e74c3c', 'TabICL': '#2ecc71'}
for name, vals in results.items():
    fpr, tpr, _ = roc_curve(y_test_flat, vals['prob_pred'])
    color = colors.get(name, '#3498db')
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {vals['AUC-ROC']:.3f})", color=color)

plt.plot([0, 1], [0, 1], 'k--', lw=1, label="Modelo aleatorio (AUC = 0.5)")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Curvas ROC - TabICL vs CatBoost')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Curvas de calibracion
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(9, 7))

for name, vals in results.items():
    prob_true, prob_pred_cal = calibration_curve(y_test_flat, vals['prob_pred'], n_bins=10)
    color = colors.get(name, '#3498db')
    ax.plot(prob_pred_cal, prob_true, marker='o', lw=2, label=name, color=color)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Calibracion perfecta')
ax.set_xlabel('Probabilidad predicha')
ax.set_ylabel('Proporcion real de defaults')
ax.set_title('Curvas de Calibracion - TabICL vs CatBoost')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusion lado a lado
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, vals) in zip(axes, results.items()):
    cm = confusion_matrix(y_test_flat, vals['class_pred'])
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Fully Paid", "Default"])
    ax.set_yticklabels(["Fully Paid", "Default"])
    ax.set_xlabel("Prediccion")
    ax.set_ylabel("Real")
    ax.set_title(f"{name}")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)
    plt.colorbar(im, ax=ax)

plt.suptitle('Matrices de Confusion', fontsize=14)
plt.tight_layout()
plt.show()

## PASO 5: Resumen y conclusiones

In [ ]:
# Resumen final
catboost_auc = results['CatBoost']['AUC-ROC']
tabicl_auc = results['TabICL']['AUC-ROC']
catboost_f1 = results['CatBoost']['F1-Score']
tabicl_f1 = results['TabICL']['F1-Score']
catboost_recall = results['CatBoost']['Recall']
tabicl_recall = results['TabICL']['Recall']

print("=" * 80)
print("RESUMEN FINAL - TabICL vs CatBoost")
print("=" * 80)
print()
print(f"  {'Metrica':<15} {'CatBoost':>10} {'TabICL':>10} {'Diferencia':>12}")
print(f"  {'-'*15} {'-'*10} {'-'*10} {'-'*12}")
print(f"  {'AUC-ROC':<15} {catboost_auc:>10.4f} {tabicl_auc:>10.4f} {(tabicl_auc-catboost_auc)*100:>+10.2f} pp")
print(f"  {'F1-Score':<15} {catboost_f1:>10.4f} {tabicl_f1:>10.4f} {(tabicl_f1-catboost_f1)*100:>+10.2f} pp")
print(f"  {'Recall':<15} {catboost_recall:>10.4f} {tabicl_recall:>10.4f} {(tabicl_recall-catboost_recall)*100:>+10.2f} pp")
print()
print("DIFERENCIAS CLAVE:")
print("  - CatBoost: modelo clasico de gradient boosting, entrena arboles sobre nuestros datos.")
print("    Requiere ajuste de hiperparametros para maximizar rendimiento.")
print("  - TabICL: foundation model pre-entrenado, funciona por in-context learning.")
print("    Sin entrenamiento ni ajuste de parametros. Un unico forward pass.")
print()
winner = "TabICL" if tabicl_auc > catboost_auc else "CatBoost"
print(f"  Mejor modelo por AUC-ROC: {winner}")